# SKAB Anomaly Detection — XGBoost & Random Forest on Combined Dataset
**Datasets:** valve1 + valve2 + other merged into one (37,459 rows, 34 files)  
**Algorithms:** XGBoost, Random Forest (supervised)  
**Evaluation:** Group 5-Fold CV grouped by dataset/file  
**Note:** Runs locally. Data loaded from relative paths alongside this notebook.

## Cell 1 — Install Dependencies

In [ ]:
!pip install -q xgboost scikit-learn pandas numpy matplotlib seaborn

## Cell 2 — Imports

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    roc_auc_score, classification_report
)

BASE_DIR = os.getcwd()
print(f'Base directory: {BASE_DIR}')

FEATURE_COLS = [
    'Accelerometer1RMS', 'Accelerometer2RMS', 'Current',
    'Pressure', 'Temperature', 'Thermocouple',
    'Voltage', 'Volume Flow RateRMS'
]

DATASETS = {
    'valve1': [f'valve1/{i}.csv' for i in range(16)],
    'valve2': [f'valve2/{i}.csv' for i in range(4)],
    'other':  [f'other/{f}' for f in sorted(os.listdir(os.path.join(BASE_DIR, 'other'))) if f.endswith('.csv')]
}

## Cell 3 — Load & Combine All Datasets

In [ ]:
def load_csv(path):
    df = pd.read_csv(path, sep=';', parse_dates=['datetime'])
    return df.sort_values('datetime').reset_index(drop=True)

# Anomaly-free baseline for scaler
baseline_df = load_csv(os.path.join(BASE_DIR, 'anomaly-free', 'anomaly-free.csv'))
print(f'Baseline shape: {baseline_df.shape}')

# Load & merge all datasets
all_dfs = []
for dataset_name, file_list in DATASETS.items():
    for fpath in file_list:
        full_path = os.path.join(BASE_DIR, fpath)
        if os.path.exists(full_path):
            df = load_csv(full_path)
            df['source_file']    = os.path.basename(fpath)
            df['source_dataset'] = dataset_name
            all_dfs.append(df)

combined = pd.concat(all_dfs, ignore_index=True)
print(f'\nCombined dataset: {len(combined)} rows, {combined["source_file"].nunique()} files')
print(f'Overall anomaly rate: {combined["anomaly"].mean():.1%}')
print('\nBreakdown by dataset:')
print(combined.groupby('source_dataset')['anomaly'].agg(['count','mean']).rename(columns={'count':'rows','mean':'anomaly_rate'}))

## Cell 4 — Feature Engineering

In [ ]:
def add_rolling_features(df, window=5):
    df = df.copy()
    for col in FEATURE_COLS:
        df[f'{col}_roll_mean'] = df[col].rolling(window, min_periods=1).mean()
        df[f'{col}_roll_std']  = df[col].rolling(window, min_periods=1).std().fillna(0)
    return df

EXTENDED_COLS = FEATURE_COLS + \
    [f'{c}_roll_mean' for c in FEATURE_COLS] + \
    [f'{c}_roll_std'  for c in FEATURE_COLS]

# Fit scaler on anomaly-free baseline only
baseline_feat = add_rolling_features(baseline_df)
scaler = StandardScaler()
scaler.fit(baseline_feat[EXTENDED_COLS])

# Apply to combined
feat_df = add_rolling_features(combined)
X       = scaler.transform(feat_df[EXTENDED_COLS])
y       = feat_df['anomaly'].values
groups  = (feat_df['source_dataset'] + '/' + feat_df['source_file']).values

print(f'Feature matrix: {X.shape}')
print(f'Unique groups (files): {len(np.unique(groups))}')

## Cell 5 — Train & Evaluate Both Models

In [ ]:
neg, pos  = (y == 0).sum(), (y == 1).sum()
scale_pos = neg / pos

models = {
    'XGBoost': XGBClassifier(
        n_estimators=300, max_depth=6,
        scale_pos_weight=scale_pos,
        eval_metric='logloss',
        random_state=42, n_jobs=-1
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=300,
        class_weight='balanced',
        random_state=42, n_jobs=-1
    )
}

gkf = GroupKFold(n_splits=5)
all_results   = {}
per_file_rows = []

for model_name, clf in models.items():
    print(f'\n=== {model_name} — Combined Dataset (Group 5-Fold CV) ===')
    all_preds  = np.zeros(len(y))
    all_probas = np.zeros(len(y))

    for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups)):
        clf.fit(X[train_idx], y[train_idx])
        all_preds[test_idx]  = clf.predict(X[test_idx])
        all_probas[test_idx] = clf.predict_proba(X[test_idx])[:, 1]
        fold_f1 = f1_score(y[test_idx], all_preds[test_idx], zero_division=0)
        print(f'  Fold {fold+1}  F1: {fold_f1:.4f}')

    print(classification_report(y, all_preds, target_names=['Normal','Anomaly'], zero_division=0))
    print(f'ROC-AUC: {roc_auc_score(y, all_probas):.4f}')

    all_results[model_name] = {
        'preds': all_preds, 'probas': all_probas,
        'f1':        round(f1_score(y, all_preds, zero_division=0), 4),
        'precision': round(precision_score(y, all_preds, zero_division=0), 4),
        'recall':    round(recall_score(y, all_preds, zero_division=0), 4),
        'roc_auc':   round(roc_auc_score(y, all_probas), 4)
    }

    for grp in sorted(np.unique(groups)):
        mask = groups == grp
        f1   = f1_score(y[mask], all_preds[mask], zero_division=0)
        ds, fname = grp.split('/')
        per_file_rows.append({
            'Dataset': ds, 'File': fname, 'Model': model_name,
            'F1': round(f1, 4), 'Anomaly Rate': round(y[mask].mean(), 3)
        })

per_file_df = pd.DataFrame(per_file_rows)

## Cell 6 — Summary Table

In [ ]:
summary_rows = []
for model_name, m in all_results.items():
    summary_rows.append({
        'Model': model_name, 'F1': m['f1'],
        'Precision': m['precision'], 'Recall': m['recall'], 'ROC-AUC': m['roc_auc']
    })

summary_df = pd.DataFrame(summary_rows)
print('\n========== COMBINED DATASET — FINAL SUMMARY ==========')
print(summary_df.to_string(index=False))

## Cell 7 — Per-File F1 Breakdown

In [ ]:
for model_name in ['XGBoost', 'Random Forest']:
    print(f'\n========== PER-FILE F1 — {model_name} ==========')
    sub = per_file_df[per_file_df['Model'] == model_name][['Dataset','File','F1','Anomaly Rate']]
    print(sub.to_string(index=False))

## Cell 8 — Bar Charts: Per-File F1 by Dataset

In [ ]:
dataset_names  = ['valve1', 'valve2', 'other']
colors_map     = {'XGBoost': '#2196F3', 'Random Forest': '#FF9800'}

fig, axes = plt.subplots(len(dataset_names), 1,
                         figsize=(14, 5 * len(dataset_names)),
                         constrained_layout=True)

for ax, dataset_name in zip(axes, dataset_names):
    subset = per_file_df[per_file_df['Dataset'] == dataset_name]
    files  = sorted(subset['File'].unique())
    x      = np.arange(len(files))
    width  = 0.35

    for i, model_name in enumerate(['XGBoost', 'Random Forest']):
        model_data = subset[subset['Model'] == model_name].set_index('File')
        f1_vals    = [model_data.loc[f, 'F1'] if f in model_data.index else 0 for f in files]
        ax.bar(x + i * width, f1_vals, width, label=model_name,
               color=colors_map[model_name], alpha=0.85)

    ax.set_xticks(x + width / 2)
    ax.set_xticklabels(files, rotation=45, ha='right', fontsize=9)
    ax.set_ylim(0, 1.15)
    ax.set_ylabel('F1 Score')
    ax.set_title(f'Per-File F1 — {dataset_name} (Combined Training)', fontweight='bold')
    ax.axhline(0.8, color='red', linestyle='--', lw=1, alpha=0.5, label='0.8 threshold')
    ax.legend()

plt.savefig('per_file_f1_combined.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: per_file_f1_combined.png')

## Cell 9 — Combined vs Separate Comparison

In [ ]:
# Separate-run results (from SKAB_XGBoost_RandomForest_AllDatasets.ipynb)
separate = {
    'XGBoost':       {'F1': 0.7141, 'Precision': 0.8025, 'Recall': 0.6485, 'ROC-AUC': 0.8063},
    'Random Forest': {'F1': 0.6696, 'Precision': 0.8199, 'Recall': 0.5724, 'ROC-AUC': 0.8003},
}

compare_rows = []
for model_name, m in all_results.items():
    sep = separate[model_name]
    compare_rows.append({
        'Model':            model_name,
        'Combined F1':      m['f1'],
        'Separate F1':      sep['F1'],
        'F1 Change':        round(m['f1'] - sep['F1'], 4),
        'Combined ROC-AUC': m['roc_auc'],
        'Separate ROC-AUC': sep['ROC-AUC'],
        'ROC-AUC Change':   round(m['roc_auc'] - sep['ROC-AUC'], 4),
    })

compare_df = pd.DataFrame(compare_rows)
print('\n========== COMBINED vs SEPARATE ==========\n')
print(compare_df.to_string(index=False))